In [ ]:
import os, sys, json, subprocess, shutil
from pathlib import Path
repo = Path(os.environ['SOUP_REPO'])
results = Path(os.environ['SOUP_RESULTS'])
print('Editable challenge repo:', repo)
print('Public data:', list((repo / 'data').glob('*.jsonl')))
print('Write your solution files under', repo / 'solution')
# Add notebook cells here to replace solution/__init__.py and your kernels.
# The prepared starter is retained when you run this template unchanged.


In [ ]:
# Offline candidate install cell. Run this cell once before the server's
# separate-process solution invocation. Requires the inspected starter checkout
# at Path(os.environ["SOUP_REPO"]), including unchanged solution/swiglu.py and
# solution/test_kernels.py. No network, dependency installation, GPU imports,
# training, tests, or apply() invocation occur in this cell.
#
# All 28 solution Python sources are embedded below. The two protected starter
# files are verified only; the other 26 are installed when missing or different.
# Unexpected files are rejected before writes. Existing __pycache__ directories
# are left untouched, as in the official whole-solution hash.


def _install_soup_candidate():
    import ast
    import base64
    import hashlib
    import io
    import json
    import os
    from pathlib import Path, PurePosixPath
    import stat
    import tempfile
    import zipfile

    expected_solution_sha256 = 'f749d20c06ad60390bace705f443067f455e0b169b5441047c040ebb3f9bd885'
    expected_archive_sha256 = '237f15c4e1df17fce2de1d1cdad869c182ae7945d8851fc04bb45bd5aa4b8561'
    protected_names = ('swiglu.py', 'test_kernels.py')
    expected_settings = {'CHUNK_TOKENS': 256,
     'COMPACT_MLP_ACTIVATIONS': True,
     'FROZEN_STORAGE': True,
     'FUSED_LORA_EPILOGUE': False,
     'FUSED_NF4': True,
     'FUSED_ROTARY': False,
     'HEAD_NUMERICS': 'native',
     'HEAD_PROJECTION': 'torch',
     'SPARSE_BACKWARD': True}
    manifest = {'__init__.py': {'bytes': 1873,
                     'sha256': 'd6556e8e878ffc7cf63680459cde21aee05cef071953b9ffadce21ba15c63435'},
     'activation_storage.py': {'bytes': 7404,
                               'sha256': '95f66325e0361ced6e0c56ad06a027c66ac8d371969dc44a5840f746dfd21ca8'},
     'frozen_storage.py': {'bytes': 8584,
                           'sha256': 'c27dc833f9b7ceb15b3f7e96169c5a6572060abead71c093aac8625636aa0fe0'},
     'head_fused_t4.py': {'bytes': 26490,
                          'sha256': 'ac75f93ce5b46eeb564afd8c74409cbbeda36804c0399d639c159bf8779b14ce'},
     'head_matmul.py': {'bytes': 4134,
                        'sha256': 'c834bc001e3c6f9449ef4f5cd60d34aa961262e27c81e98e9a47186af5ec7e39'},
     'head_matmul_t4.py': {'bytes': 9051,
                           'sha256': 'b29ed64e0d1537f6682d2b75d5a979c8269030a306aa8f1dee263bb9e3be3134'},
     'integration.py': {'bytes': 12622,
                        'sha256': '3da67233c3917e5efc5e1811f096664a95e8500007533e5bb1f921c2bcbb7a2d'},
     'lora_epilogue.py': {'bytes': 9912,
                          'sha256': '124f3ff31c7f1328bfa2a0e7760fd6bd7b48d9daa654d1addef154760767964e'},
     'metric_transfer.py': {'bytes': 6052,
                            'sha256': '2b30a1c1ec61eca8841a9f2f970588f65ce2847b6b178ebd7d950995dba5a2d9'},
     'nf4.py': {'bytes': 10430,
                'sha256': '9e07a73d45f7b7119fc1f5d1d3422351e5d34e2fbd9612f7b90f56bc17ad697a'},
     'rmsnorm.py': {'bytes': 5869,
                    'sha256': 'aa55a21fd921d2f5478444b03a32694a329bb10bbbe4f3bf46e4e78f2804a93c'},
     'rotary.py': {'bytes': 13338,
                   'sha256': '8e8ef322d70eba6ac898a040a14d74b9dc8e276b3f93bd033414eeaf927b5429'},
     'settings.py': {'bytes': 1489,
                     'sha256': '17261889d0b049ddc41a2607e7106dc62e466069466344bcab885be84f9494de'},
     'streamed_head.py': {'bytes': 27782,
                          'sha256': '65ad6cada653b0d03d70375570b132c69f47a52ee8ff65ef0cb11042a63c9f19'},
     'swiglu.py': {'bytes': 3185,
                   'sha256': 'e665910faeca6091c958ed7a191604484a44216f51d6fa069f5f48220e7472e6'},
     'test_activation_storage.py': {'bytes': 14607,
                                    'sha256': '4a4b8b9819c26745fb10ef8228119981fec75012b140844c43552f0ac42291cf'},
     'test_frozen_storage.py': {'bytes': 11574,
                                'sha256': '87c98888cdb260d6e0e3e2c639ab5e9107a26da1708b8b005a2a4aa1d61ef07d'},
     'test_head_fused_t4.py': {'bytes': 24883,
                               'sha256': '335d684aa6fdc063d478265209e6ee05fcdf32fa7babe8d3e6c24644679d512d'},
     'test_head_matmul.py': {'bytes': 2743,
                             'sha256': '1ae409f55282fbd96d3c78845ab22cded87bf453149fbecbbfe7e3f62ec4c9bd'},
     'test_head_matmul_t4.py': {'bytes': 7100,
                                'sha256': '75931a45b9623e2ceb458904166218d31ffc68ccb08f129da5691cbdc961467d'},
     'test_integration.py': {'bytes': 30248,
                             'sha256': 'e8b5c5571d8baa987e38affa925defbf10de29ef025771fa20c74d41101e6aea'},
     'test_kernels.py': {'bytes': 1300,
                         'sha256': 'a45b842a4f22c1aefe404baeff1ac26511253452ecb0f633c985fbc9df8b252a'},
     'test_lora_epilogue.py': {'bytes': 15228,
                               'sha256': 'c7394c871558fe12ab684aca5e5872ecd18763467e2ace460ff9016c7efd6421'},
     'test_nf4.py': {'bytes': 10862,
                     'sha256': '2ee2f46a3011ab76c61675bef3b3e1ce40e0964d1737c92c36b71e8fd05c9a4f'},
     'test_rmsnorm.py': {'bytes': 3634,
                         'sha256': 'a0ef2638f9d7704c5da4154dcf9eac2b87069952fd3ff743da63ea84ddbdfded'},
     'test_rotary.py': {'bytes': 18180,
                        'sha256': 'a8659bbb8455c6f20868672a7a41e4c96c67b5a066cd443addd0067acf4faf37'},
     'test_streamed_head.py': {'bytes': 29925,
                               'sha256': '5ebbca158118591e39fca8b70ecf9f2485b287f07715573ab1c271acf21775d4'},
     'test_training_parity.py': {'bytes': 3303,
                                 'sha256': '412b9ebcd7d1d95c27b0e388a43f9b71820c3959ed5caa04345c09ba9fa945f1'}}
    payload_b64 = '...'

    # Decode in memory; never extract an archive directly onto the filesystem.
    archive_bytes = base64.b64decode(payload_b64, validate=True)
    if hashlib.sha256(archive_bytes).hexdigest() != expected_archive_sha256:
        raise RuntimeError("Embedded candidate archive checksum mismatch")
    payload = {}
    with zipfile.ZipFile(io.BytesIO(archive_bytes), "r") as archive:
        members = archive.infolist()
        names = [member.filename for member in members]
        if len(names) != len(set(names)) or set(names) != set(manifest):
            raise RuntimeError("Embedded archive has missing, duplicate, or unexpected entries")
        for member in members:
            name = member.filename
            path = PurePosixPath(name)
            if (path.is_absolute() or len(path.parts) != 1 or path.name != name
                    or "\\" in name or not name.endswith(".py")
                    or member.is_dir() or stat.S_ISLNK(member.external_attr >> 16)):
                raise RuntimeError("Unsafe embedded archive path: " + repr(name))
            expected = manifest[name]
            if member.file_size != expected["bytes"]:
                raise RuntimeError("Embedded source size mismatch: " + name)
            content = archive.read(member)
            if (len(content) != expected["bytes"]
                    or hashlib.sha256(content).hexdigest() != expected["sha256"]):
                raise RuntimeError("Embedded source checksum mismatch: " + name)
            compile(content, "solution/" + name, "exec")  # Parse only; execute nothing.
            payload[name] = content

    # Check settings and the no-argument entry point without importing solution.
    settings = {}
    for node in ast.parse(payload["settings.py"]).body:
        if isinstance(node, ast.Assign):
            for lhs in node.targets:
                if isinstance(lhs, ast.Name):
                    settings[lhs.id] = ast.literal_eval(node.value)
    if settings != expected_settings:
        raise RuntimeError("Embedded candidate settings differ from the fixed candidate")
    entrypoints = [node for node in ast.parse(payload["__init__.py"]).body
                   if isinstance(node, ast.FunctionDef) and node.name == "apply"]
    if len(entrypoints) != 1:
        raise RuntimeError("Embedded candidate must define exactly one apply()")
    args = entrypoints[0].args
    if args.posonlyargs or args.args or args.kwonlyargs or args.vararg or args.kwarg:
        raise RuntimeError("Embedded candidate apply() must take no arguments")

    repo_value = os.environ.get("SOUP_REPO")
    if repo_value is None or not repo_value.strip():
        raise RuntimeError("Set SOUP_REPO to the existing official starter checkout first")
    repo = Path(repo_value).resolve(strict=True)
    if not repo.is_dir():
        raise RuntimeError("SOUP_REPO must name an existing directory")
    solution = repo / "solution"
    if solution.is_symlink() or not solution.is_dir():
        raise RuntimeError("SOUP_REPO/solution must be an existing real directory")

    # Preflight every destination before any write. Never delete unknown files.
    unexpected = []
    for path in sorted(solution.rglob("*")):
        relative = path.relative_to(solution)
        name = str(relative)
        if path.is_symlink():
            raise RuntimeError("Refusing a symlink in solution/: " + name)
        if "__pycache__" in relative.parts:
            continue  # Official hash excludes these; do not modify or remove them.
        if not path.is_file() or name not in manifest:
            unexpected.append(name)
    if unexpected:
        raise RuntimeError("Unexpected solution/ entries; inspect them before retrying: "
                           + ", ".join(unexpected))
    for name in protected_names:
        path = solution / name
        if not path.is_file() or path.read_bytes() != payload[name]:
            raise RuntimeError("Protected starter source missing or different: solution/"
                               + name + "; no files were written")

    updated = []
    unchanged = []
    for name, content in sorted(payload.items()):
        destination = solution / name
        if name in protected_names:
            continue  # Verify-only, even when the bytes already match.
        if destination.is_file() and destination.read_bytes() == content:
            unchanged.append(name)
            continue
        # Atomic replacement avoids partially written source files and does not
        # follow a destination hardlink. Temporary files stay inside solution/.
        temporary = None
        try:
            with tempfile.NamedTemporaryFile(mode="wb", dir=solution,
                                             prefix="._soup_install_", suffix=".tmp",
                                             delete=False) as handle:
                temporary = Path(handle.name)
                handle.write(content)
                handle.flush()
                os.fchmod(handle.fileno(), 0o644)
            os.replace(temporary, destination)
            temporary = None
            updated.append(name)
        finally:
            if temporary is not None:
                try:
                    temporary.unlink()  # Only the temporary file created above.
                except FileNotFoundError:
                    pass

    # Same algorithm as unchanged harness/check_rules.py:solution_sha256.
    # It hashes relative path bytes followed by file bytes in sorted path order.
    digest = hashlib.sha256()
    for path in sorted(Path(solution).rglob("*")):
        if path.is_file() and "__pycache__" not in path.parts:
            digest.update(str(path.relative_to(solution)).encode())
            digest.update(path.read_bytes())
    observed_solution_sha256 = digest.hexdigest()
    if observed_solution_sha256 != expected_solution_sha256:
        raise RuntimeError("Installed whole-solution hash mismatch: "
                           + observed_solution_sha256)
    print(json.dumps({
        "status": "installed_and_verified",
        "solution_sha256": observed_solution_sha256,
        "source_files": len(manifest),
        "updated_files": len(updated),
        "already_matching_files": len(unchanged),
        "protected_files_verified": list(protected_names),
        "settings": settings,
        "apply_called": False,
    }, indent=2, sort_keys=True))


_install_soup_candidate()


In [ ]:
# Research A/B: original masked scatter instead of zeros_like + index_copy.
# Same 5,963-MiB source overlay and same MLP active-row decisions as the
# observed 65.43-second candidate; only its dense MLP output scatter differs.
from pathlib import Path
import hashlib, json
init_path = repo / "solution" / "__init__.py"
init_src = init_path.read_text(encoding="utf-8")
assert hashlib.sha256(init_src.encode()).hexdigest() == "d6556e8e878ffc7cf63680459cde21aee05cef071953b9ffadce21ba15c63435"
padding_src = "\"\"\"Within-batch Qwen2 MLP padding compaction with an original fused scatter.\n\nThe existing attention mask determines which MLP rows are not padding. We keep\nall original valid positions in original row order and restore zeros for padding.\nAttention, dropout, batch membership, positions and labels remain unchanged.\nThe original GEMMs and SwiGLU still execute on compact active rows.\n\"\"\"\n\nimport contextvars\nimport inspect\nimport torch\nimport triton\nimport triton.language as tl\n\n_ACTIVE_ROWS = contextvars.ContextVar(\"_soup_mlp_active_rows\", default=None)\n\n\n@triton.jit\ndef soup_mlp_scatter_dense(COMPACT, ROW_MASK, PREFIX, OUTPUT,\n                           WIDTH: tl.constexpr, BLOCK: tl.constexpr):\n    row = tl.program_id(0)\n    col = tl.arange(0, BLOCK)\n    keep = tl.load(ROW_MASK + row)\n    idx = tl.maximum(tl.load(PREFIX + row), 0)\n    value = tl.load(COMPACT + idx * WIDTH + col, mask=keep & (col < WIDTH), other=0.0)\n    tl.store(OUTPUT + row * WIDTH + col, value, mask=col < WIDTH)\n\n\nclass _ScatterCompacted(torch.autograd.Function):\n    @staticmethod\n    def forward(ctx, compact_output, active, row_mask, prefix, nrows):\n        width = compact_output.shape[-1]\n        out = torch.empty((nrows, width), device=compact_output.device, dtype=compact_output.dtype)\n        soup_mlp_scatter_dense[(nrows,)](\n            compact_output, row_mask, prefix, out,\n            WIDTH=width, BLOCK=triton.next_power_of_2(width),\n            num_warps=4,\n        )\n        ctx.save_for_backward(active)\n        ctx.set_materialize_grads(False)\n        return out\n\n    @staticmethod\n    def backward(ctx, grad_out):\n        if grad_out is None:\n            return None, None, None, None, None\n        (active,) = ctx.saved_tensors\n        return grad_out.index_select(0, active), None, None, None, None\n\n\ndef patch_mlp_padding_compaction():\n    from transformers.models.qwen2 import modeling_qwen2\n\n    mlp_cls = modeling_qwen2.Qwen2MLP\n    model_cls = modeling_qwen2.Qwen2Model\n    if getattr(mlp_cls.forward, \"_soup_padding_compaction\", False):\n        return True\n\n    original_mlp = mlp_cls.forward\n    original_model = model_cls.forward\n    model_signature = inspect.signature(original_model)\n\n    def model_forward(self, *args, **kwargs):\n        try:\n            bound = model_signature.bind_partial(self, *args, **kwargs)\n            mask = bound.arguments.get(\"attention_mask\")\n        except TypeError:\n            mask = kwargs.get(\"attention_mask\")\n\n        plan = None\n        if (isinstance(mask, torch.Tensor) and mask.is_cuda and mask.ndim == 2\n                and mask.numel() > 0):\n            flat_mask = (mask.reshape(-1) != 0).contiguous()\n            active = torch.nonzero(flat_mask, as_tuple=False).flatten()\n            if 0 < active.numel() < flat_mask.numel():\n                # One short prefix scan per batch; reused in all decoder MLPs.\n                prefix = torch.cumsum(flat_mask.to(torch.int32), dim=0) - 1\n                plan = (active, tuple(mask.shape), flat_mask, prefix)\n\n        token = _ACTIVE_ROWS.set(plan)\n        try:\n            return original_model(self, *args, **kwargs)\n        finally:\n            _ACTIVE_ROWS.reset(token)\n\n    def mlp_forward(self, x):\n        plan = _ACTIVE_ROWS.get()\n        if (plan is None or not isinstance(x, torch.Tensor) or not x.is_cuda\n                or x.ndim != 3):\n            return original_mlp(self, x)\n\n        active, mask_shape, flat_mask, prefix = plan\n        if (tuple(x.shape[:2]) != mask_shape or active.device != x.device\n                or not x.is_contiguous()):\n            return original_mlp(self, x)\n\n        flat = x.reshape(-1, x.shape[-1])\n        compact = flat.index_select(0, active)\n        compact_out = original_mlp(self, compact)\n        if (compact_out.ndim != 2 or compact_out.shape[0] != active.numel()\n                or not compact_out.is_contiguous() or compact_out.shape[1] > 8192):\n            out = torch.zeros_like(flat)\n            out.index_copy_(0, active, compact_out.reshape(active.numel(), -1))\n        else:\n            out = _ScatterCompacted.apply(\n                compact_out, active, flat_mask, prefix, flat.shape[0],\n            )\n        return out.view(*x.shape[:-1], compact_out.shape[-1])\n\n    model_forward._soup_padding_compaction = True\n    mlp_forward._soup_padding_compaction = True\n    model_cls.forward = model_forward\n    mlp_cls.forward = mlp_forward\n    return True\n"
padding_path = repo / "solution" / "mlp_padding.py"
compile(padding_src, str(padding_path), "exec")
padding_path.write_text(padding_src, encoding="utf-8")
anchor = """    if COMPACT_MLP_ACTIVATIONS:
        from .activation_storage import patch_compact_mlp_activations
        patch_compact_mlp_activations()
"""
replacement = anchor + """    from .mlp_padding import patch_mlp_padding_compaction
    patch_mlp_padding_compaction()
"""
assert init_src.count(anchor) == 1
init_src = init_src.replace(anchor, replacement, 1)
compile(init_src, str(init_path), "exec")
init_path.write_text(init_src, encoding="utf-8")
digest = hashlib.sha256()
for path in sorted((repo / "solution").rglob("*")):
    if path.is_file() and "__pycache__" not in path.parts:
        digest.update(str(path.relative_to(repo / "solution")).encode())
        digest.update(path.read_bytes())
record = {
    "experiment": "mlp_padding_fused_scatter",
    "prior_validated_candidate": "chatgpt-storage-mlp-padding-20261031-v1-r40",
    "solution_sha256": digest.hexdigest(),
    "mlp_padding_source_sha256": hashlib.sha256(padding_src.encode()).hexdigest(),
    "original_kernel": "soup_mlp_scatter_dense",
    "protected_paths_unchanged": True,
}
(results / "mlp_scatter_source.json").write_text(json.dumps(record, indent=2) + "\n")
print(record)


In [ ]:
# Isolated low-memory head chunk A/B. All 28 original source files and the
# original masked-scatter Triton MLP patch are kept unchanged except this
# source-guarded fixed CHUNK_TOKENS setting (256 -> 128).
import os, hashlib
from pathlib import Path
_settings = Path(os.environ["SOUP_REPO"]) / "solution" / "settings.py"
_before = _settings.read_bytes()
assert hashlib.sha256(_before).hexdigest() == "17261889d0b049ddc41a2607e7106dc62e466069466344bcab885be84f9494de", "Unexpected source settings; abort"
_old = b"CHUNK_TOKENS = 256\n"
assert _before.count(_old) == 1
_after = _before.replace(_old,b"CHUNK_TOKENS = 128\n")
_settings.write_bytes(_after)
assert _settings.read_bytes() == _after
print("bounded native head chunk 128; settings sha256=",hashlib.sha256(_after).hexdigest())


In [ ]:
# Run only the solution. The server reuses its saved matching public baseline.
run = Path('/kaggle/working/training-run')
subprocess.run([sys.executable, str(repo / 'harness/run_task.py'), '--out', str(run), '--timeout', '900', '--only', 'solution'], cwd=repo, check=True)


In [ ]:
# Export small reports and logs. Large adapter/checkpoint files remain in this job's VM workspace.
for path in run.rglob('*'):
    if path.is_file() and path.suffix in ('.json', '.jsonl', '.log', '.yaml') and path.stat().st_size < 4 * 1024**2:
        target = results / path.relative_to(run)
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(path, target)
print((run / 'result.json').read_text())
# This is a public-data screen. Notebook output does not establish official admission or score.


In [ ]:
# Post-run official-source kernel-participation audit.
# These checks do NOT modify measured training or the task, and run only after
# the unchanged solution-only benchmark finished and exported its results.
import json, runpy, os
from pathlib import Path
_checker = runpy.run_path(str(repo / "harness" / "check_rules.py"),
                          run_name="_source_attribution_only")
_own, _copies, _lookalikes, _banned, _broken = _checker["source_kernels"](repo / "solution")
_result=json.loads((run / "result.json").read_text())
_sol_runs=_result["runs"]["solution"]
assert len(_sol_runs)==1 and _sol_runs[0].get("exit_code")==0
_met=_sol_runs[0]["metrics"]
_files=_met.get("triton_files", {})
_launches=_met.get("triton_launches", {})
_kernel_ms=_met.get("own_kernel_ms", {})
_allow=[
    n for n in _own if n in _launches and _launches[n]>0 and
    str(_files.get(n,"")).replace("\\","/").startswith("solution/")
]
_credited={n:float(_kernel_ms[n]) for n in _allow if n in _kernel_ms}
_timed=float(_met["timed_loop_s"])
assert _timed>0
_share=sum(_credited.values())/(_timed*1000.)
_audit={
    "kind":"pinned_official_source_kernel_timing_steps3plus",
    "useful_source_files":{n:_own[n] for n in _allow},
    "credited_kernel_ms":_credited,
    "timed_loop_s":_timed,
    "credited_original_share_percent":100*_share,
    "passed_minimum_5pct":_share>=0.05,
    "example_kernels_excluded":_copies,
    "near_example_kernels":_lookalikes,
    "banned_source_imports":_banned,
    "source_parse_errors":_broken,
    "source":"harness/check_rules.py; harness/_train.py",
    "note":"Single public solution run; not hidden jury or paired baseline admission.",
}
(results/"original_kernel_share.json").write_text(json.dumps(_audit,indent=2)+"\n")
print("ORIGINAL_TRITON_SHARE_PERCENT",round(100*_share,5))
print("ORIGINAL_TRITON_KERNELS",json.dumps(_credited))
print("TRITON_SOURCE_IMPORT_ISSUES", json.dumps(_banned+_broken))


In [ ]:
# After, NOT DURING, the unchanged measured 64-step Soup command:
# run the real 79 pinned Soup regressions with an offline, test-only Pytest bundle.
# This does NOT install packages, modify SOUP_REPO/solution, change task/switches,
# access hidden rows, or count its execution in the measured training command.
# The included pure-Python test dependencies are pytest 9.0.2, pluggy 1.6.0,
# packaging 25.0, iniconfig 2.3.0 and selected Pygments 2.20.0 Python sources,
# with their license texts under THIRD_PARTY_LICENSES/ in the ZIP.
import base64, hashlib, json, os, re, subprocess, sys
from pathlib import Path
_bundle_base64 = "..."
_bundle_expected_sha256 = "d17ccdad40df9b31be4a1ceec7d95dd45df54c877dfe7aa1b41f9be92cf23cef"
_bundle = Path("/kaggle/working/soup_test_only_pytest_bundle.zip")
_zipbytes = base64.b64decode(_bundle_base64, validate=True)
assert hashlib.sha256(_zipbytes).hexdigest() == _bundle_expected_sha256
_bundle.write_bytes(_zipbytes)
_testenv = dict(
    os.environ,
    PYTHONPATH=str(_bundle) + os.pathsep + os.environ.get("PYTHONPATH",""),
    PYTEST_DISABLE_PLUGIN_AUTOLOAD="1",
    PYTHONDONTWRITEBYTECODE="1",
    SOUP_APPLY_SOLUTION="1",
    PY_COLORS="0",
)
_cmd = [sys.executable,"-m","pytest","tests","-q","-p","no:cacheprovider","--color=no","--tb=short"]
_p = subprocess.run(_cmd, cwd=repo, env=_testenv, text=True,
                    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                    check=False,timeout=1600)
_log = _p.stdout
(results / "soup79_bundle_test.log").write_text(_log[-300000:],encoding="utf-8")
_summary = {
    "kind":"postrun_unmodified_official_test_sources_with_bundled_pytest",
    "training_benchmark_unchanged": True,
    "bundle_sha256":_bundle_expected_sha256,
    "command":"PYTHONPATH=bundled_pytest SOUP_APPLY_SOLUTION=1 python -m pytest tests -q -p no:cacheprovider --color=no --tb=short",
    "returncode":_p.returncode,
    "exact_79_passed":bool(re.search(r"(?<!\d)79 passed\b",_log)) and _p.returncode==0,
    "tail":_log[-12000:],
}
(results / "soup79_bundle_test.json").write_text(json.dumps(_summary,indent=2)+"\n",encoding="utf-8")
print("BUNDLED_79_REGRESSIONS_RETURN_CODE:",_p.returncode)
print(_log[-3000:])
if not _summary["exact_79_passed"]:
    raise RuntimeError("OFFICIAL_79_TEST_SUITE_NOT_PASSED: " + "\n".join(_log.splitlines()[-36:])[-3600:])
print("79 official Soup regression tests passed with exact source and enabled solution.apply().")


In [ ]:
# Both postrun validation checks must pass before classing this job as a
# candidate with preliminary numerical/regression and original-kernel evidence.
if _banned or _broken:
    raise RuntimeError("OWN_SOURCE_DISQUALIFICATION: "+repr(_banned+_broken))
if _share < 0.05:
    raise RuntimeError("OWN_TRITON_SHARE_LT_5_PERCENT: "+
                       str(round(100*_share,5))+
                       "% measured steps3+; required >=5%")
print("PRELIMINARY_POSTRUN_TESTS_PASS: 79 Soup regressions and "+
      str(round(100*_share,4))+"% original Triton share")
